# Lesson 23 — BLE 服务模型与 provisioning — Solution

这是独立答案 Notebook。先完成主课再打开它。

In [ ]:
from pathlib import Path
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tools/notebook_setup.py").is_file())
exec((project / "tools/notebook_setup.py").read_text())

In [ ]:
%%academy_lab 23-ble-provisioning solution exercise pass
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& device) : device_(device) {
        }
        void tick();

    private:
        Device& device_;
        bool provisioned_ = false;
    };
}

namespace academy {
    void Controller::tick() {
        // `text` is a model of a validated GATT characteristic write, not radio traffic.
        if (!device_.input.connected) {
            device_.output.state = "advertising";
            return;
        }
        if (device_.input.text.rfind("wifi:", 0) == 0 &&
            device_.input.text.size() > 5) {
            provisioned_ = true;
            device_.output.outgoing = "provisioning-status:accepted";
            ++device_.output.events;
        } else if (!device_.input.text.empty()) {
            device_.output.error = true;
            device_.output.outgoing = "provisioning-status:rejected";
        }
        device_.output.state = provisioned_ ? "provisioned" : "connected";
        device_.output.display =
            "Service: Provisioning / Characteristic: WiFiCredentials";
    }
}
